# 05 · Forecast models: baselines, ridge, boosting and MLP (US Embassy station, OpenAQ 8160)

**Goals.** H1: does the selected neural network have positive MAE skill over persistence on smog-season days? (Here on the validation folds; the test answer comes once, on 23 November.) H2: do weather features improve the network's validation MAE, on identical rows?

**Protocol.**
1. Fit on every complete training row from 2019-07-01, all months.
2. Score on smog-season (Nov–Feb) target days only.
3. Select on the mean MAE of the two rolling folds (fold 1 validates 2021/22, fold 2 validates 2023/24; 2022/23 is an AirNow outage).
4. The test winter (2024/25) is sealed: nothing here fits on, scores or plots its PM2.5.

Fold 1 fits on about 170 valid smog days (fewer once every feature must be present), so selection is noisier there.

All logic lives in `src/uaq/models.py`; this notebook only calls it.

In [ ]:
# Setup: works whether or not you ran `pip install -e .`
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from uaq.config import load_config, ROOT
from uaq import features, metrics, models, weather
from uaq.periods import in_months, season_label, season_year
cfg = load_config()
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
%matplotlib inline
print("Project root:", ROOT)

## 1 · Features, the missingness rule and the ablation rows
The rule was fixed before any model was trained (CLAUDE.md, rule 7): on training rows where both `y` and `pm_lag0` are present, drop any feature missing on more than 10%; `pm_lag0` is never dropped. The result is recorded in `features.drop_columns`, and the cell below stops if the config and the rule disagree.

In [ ]:
feat = pd.read_csv(ROOT / "data" / "processed" / "features_8160.csv", parse_dates=["date", "target_date"])
share, rule_drop = models.missingness_rule(feat, cfg)
display(share.round(3).to_frame("missing share (training rows with y and pm_lag0)"))
drop = cfg["features"]["drop_columns"]
assert set(rule_drop) <= set(drop), "features.drop_columns disagrees with the missingness rule: update the config"
print("Dropped by the missingness rule:", rule_drop)
print("All dropped features (config):", drop)

p = cfg["forecast"]["periods"]
tr_days = feat[(feat["target_date"] >= p["core"]["start"]) & (feat["target_date"] <= p["split"]["train"]["end"])]
print(f"Training target days with no valid daily mean (station outages; excluded, never imputed): "
      f"{tr_days['y'].isna().mean():.1%} of {len(tr_days):,}")

In [ ]:
cols_w = features.feature_columns(feat, weather=True, drop=drop)
cols_nw = features.feature_columns(feat, weather=False, drop=drop)
FEATS = {"weather": cols_w, "no weather": cols_nw}
ablation = features.complete_rows(feat, cols_w)   # rows complete WITH weather: used for both feature sets
print(f"{len(cols_w)} features with weather: {cols_w}")
print(f"{len(cols_nw)} features without weather: {cols_nw}")

folds_w, folds_nw = models.fold_rows(ablation, cols_w, cfg), models.fold_rows(ablation, cols_nw, cfg)
assert all(a[1].index.equals(b[1].index) and a[2].index.equals(b[2].index) for a, b in zip(folds_w, folds_nw))
display(pd.DataFrame([{"fold": k, "fit rows": len(fit), "of which smog": int(fit["smog"].sum()),
                       "fit from": fit["target_date"].min().date(), "fit to": fit["target_date"].max().date(),
                       "score rows (smog)": len(sc), "score from": sc["target_date"].min().date(),
                       "score to": sc["target_date"].max().date()} for k, fit, sc in folds_w]).set_index("fold"))

## 2 · Event threshold and the baselines
Threshold from training smog-season daily means, as in notebook 01.

In [ ]:
thr_name, thr, thr_why = models.event_threshold(feat, cfg)
print(f"Event threshold: {thr_name} = {thr} µg/m³. {thr_why}")
q = cfg["forecast"]["models"]["extreme_quantile"]

BASE = {"persistence": models.Persistence, "climatology": models.Climatology}
base_fold, _ = models.select(ablation, cols_w, BASE, cfg, thr)
SHOW_FOLD = ["candidate", "fold", "n", "mae", "rmse", "skill_mae", "events", "precision", "recall", "f1", "missed"]
display(base_fold[SHOW_FOLD].round(3))

## 3 · Ridge and gradient boosting, with and without weather
Candidates are listed simplest first, so a tie (equal to 0.01 µg/m³) goes to the simpler model.

In [ ]:
CANDS = {"ridge": models.ridge_candidates(cfg), "hgb": models.hgb_candidates(cfg), "mlp": models.mlp_candidates(cfg)}
runs = {}
SHOW = ["mean_mae", "rmse", "skill_mae", "f1", "missed", "chosen"]
for fam in ("ridge", "hgb"):
    for fs, cols in FEATS.items():
        runs[(fam, fs)] = models.select(ablation, cols, CANDS[fam], cfg, thr)
        display(Markdown(f"**{fam}, {fs}**")); display(runs[(fam, fs)][1][SHOW].round(3))

## 4 · MLP (5-seed ensembles), with and without weather

In [ ]:
for fs, cols in FEATS.items():
    runs[("mlp", fs)] = models.select(ablation, cols, CANDS["mlp"], cfg, thr)
    display(Markdown(f"**mlp, {fs}**")); display(runs[("mlp", fs)][1][SHOW].round(3))

chosen = {k: s.index[s["chosen"]][0] for k, (_, s) in runs.items()}
with pd.option_context("display.max_colwidth", 100):
    display(pd.Series({f"{f} ({fs})": c for (f, fs), c in chosen.items()}).to_frame("chosen config"))

In [ ]:
# Seed spread of the chosen MLP (with weather): MAE of each seed and of the 5-seed mean
seeds = cfg["forecast"]["models"]["seeds"]
mlp_runs = list(models.run_folds(ablation, cols_w, CANDS["mlp"][chosen[("mlp", "weather")]], cfg))
rows = []
for r in mlp_runs:
    per_seed = r["model"].predict_per_seed(r["score_df"].set_index("target_date")[cols_w])
    for s, pr in zip(seeds, per_seed):
        rows.append({"fold": r["fold"], "seed": s, "mae": metrics.regression(r["score_df"]["y"], pr)["mae"]})
    rows.append({"fold": r["fold"], "seed": "5-seed mean", "mae": metrics.regression(r["score_df"]["y"], r["pred"])["mae"]})
spread = pd.DataFrame(rows).pivot(index="fold", columns="seed", values="mae")
spread["seed std"] = spread[seeds].std(axis=1)
display(spread.round(2))

In [ ]:
# Training vs early-stopping loss, fold 2, seed 0 (standardised log1p scale)
m0 = mlp_runs[-1]["model"].models_[0]
h = pd.DataFrame(m0.history_)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(h["epoch"], h["train"], label="train (first 85% of fit rows)")
ax.plot(h["epoch"], h["early_stop"], label="early stop (last 15% of fit rows)")
ax.axvline(m0.best_epoch_, color="grey", ls="--", label=f"best epoch {m0.best_epoch_}")
ax.set_xlabel("epoch"); ax.set_ylabel("MSE"); ax.legend(fontsize=8); ax.set_title("MLP loss, fold 2, seed 0")
fig.tight_layout(); plt.show()

## 5 · Summary of the chosen models
Skill is 1 − MAE / persistence MAE on the same rows (positive = better than persistence).

In [ ]:
parts = [base_fold]
for (fam, fs), name in chosen.items():
    pf = runs[(fam, fs)][0]
    parts.append(pf[pf["candidate"] == name].assign(candidate=f"{fam} ({fs})"))
final_fold = pd.concat(parts, ignore_index=True).rename(columns={"candidate": "model"})
display(final_fold[["model", "fold", "n", "mae", "rmse", "skill_mae", "precision", "recall", "f1", "missed", "events"]].round(3))

final = (final_fold.groupby("model", sort=False)
         .agg(mae=("mae", "mean"), rmse=("rmse", "mean"), skill_mae=("skill_mae", "mean"),
              precision=("precision", "mean"), recall=("recall", "mean"), f1=("f1", "mean"),
              missed=("missed", "sum"), events=("events", "sum")))
display(Markdown("**Mean over the two folds** (missed and events summed)")); display(final.round(3))

In [ ]:
order = final["mae"].sort_values()
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(order.index, order.values, color=["#C44E52" if m == "persistence" else "#4C72B0" for m in order.index])
ax.axvline(final.loc["persistence", "mae"], color="#C44E52", ls="--", label="persistence")
ax.invert_yaxis(); ax.set_xlabel("mean MAE over 2 folds (µg/m³)"); ax.legend(fontsize=8)
ax.set_title("Validation smog-season MAE by model")
fig.tight_layout(); plt.show()

## 6 · Worst days
Rows whose observed PM2.5 is above the 90th percentile of that fold's scored rows. Bias = mean(prediction − observed): negative means underprediction.

In [ ]:
worst = final_fold[["model", "fold", "n_extreme", "mae_extreme", "bias_extreme", "skill_mae_extreme"]]
display(worst.round(2))
wm = worst.groupby("model", sort=False)[["mae_extreme", "bias_extreme", "skill_mae_extreme"]].mean()
display(wm.round(2))
learned = wm.drop(index=["persistence", "climatology"])
under = learned.index[learned["bias_extreme"] < 0].tolist()
over = learned.index[learned["bias_extreme"] >= 0].tolist()
beaten = learned.index[learned["skill_mae_extreme"] < 0].tolist()
pw = wm.loc["persistence"]
display(Markdown(
    f"**Worst days:** {len(under)} of {len(learned)} chosen models underpredict the worst days on average "
    f"({', '.join(under) or 'none'})" + (f"; {', '.join(over)} overpredict" if over else "") + ". "
    f"Persistence (itself biased {pw['bias_extreme']:+.0f} µg/m³) has a lower worst-day MAE "
    f"({pw['mae_extreme']:.0f} µg/m³) than {len(beaten)} of {len(learned)} chosen models"
    + (", so persistence wins on the worst days." if len(beaten) == len(learned) else ".")))

## 7 · Weather ablation (H2): identical rows, each feature set with its own selected configuration
Difference = MAE with weather − MAE without (negative = weather helps).

In [ ]:
abl = []
for fam in ("ridge", "mlp"):
    w = final_fold[final_fold["model"] == f"{fam} (weather)"].set_index("fold")
    nw = final_fold[final_fold["model"] == f"{fam} (no weather)"].set_index("fold")
    assert (w["n"] == nw["n"]).all(), "ablation rows differ"
    for k in w.index:
        abl.append({"model": fam, "fold": k, "n": int(w.loc[k, "n"]), "mae weather": w.loc[k, "mae"],
                    "mae no weather": nw.loc[k, "mae"], "difference": w.loc[k, "mae"] - nw.loc[k, "mae"]})
abl = pd.DataFrame(abl)
display(abl.round(2))
mean_diff = abl.groupby("model")["difference"].mean()
display(Markdown("**Mean difference over folds:** " + "; ".join(f"{m} {d:+.2f} µg/m³" for m, d in mean_diff.items())))

## 8 · Test-winter mildness check (ERA5 weather only, no PM2.5)
Is 2024/25 a mild winter compared with the training winters? Only weather is used; the test PM2.5 stays sealed.

In [ ]:
wd = weather.to_daily(weather.load_hourly(cfg))
w = wd[in_months(wd["date"], cfg["smog_months"]).values]
w = w.assign(winter=season_year(w["date"]).values)
w = w[w["winter"].between(2019, 2024)]
mild = w.groupby("winter").agg(days=("date", "size"), temp_mean=("temp_mean", "mean"), temp_min=("temp_min", "mean"),
                               temp_range=("temp_range", "mean"), calm_hours=("calm_hours", "mean"))
mild["role"] = ["train" if s <= 2022 else "validation" if s == 2023 else "test" for s in mild.index]
mild.index = [season_label(s) for s in mild.index]
display(mild.round(2))

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].plot(mild.index, mild["temp_mean"], marker="o", label="mean")
axes[0].plot(mild.index, mild["temp_min"], marker="o", label="daily minimum")
axes[0].set_ylabel("°C, Nov–Feb mean"); axes[0].legend(fontsize=8)
axes[1].bar(mild.index, mild["calm_hours"], color=["#4C72B0" if r == "train" else "#DD8452" for r in mild["role"]])
axes[1].set_ylabel("calm hours per day")
for ax in axes:
    ax.tick_params(axis="x", labelrotation=45)
fig.suptitle("Winter weather, ERA5 (orange = validation and test)"); fig.tight_layout(); plt.show()

tr, te = mild[mild["role"] == "train"], mild[mild["role"] == "test"].iloc[0]
warmer = int((te["temp_mean"] > tr["temp_mean"]).sum())
display(Markdown(
    f"**Mildness:** 2024/25 averaged {te['temp_mean']:.1f} °C (daily minimum {te['temp_min']:.1f} °C, "
    f"{te['calm_hours']:.1f} calm hours a day), warmer than {warmer} of {len(tr)} training winters "
    f"({tr['temp_mean'].min():.1f} to {tr['temp_mean'].max():.1f} °C). "
    + ("It is milder than every training winter." if warmer == len(tr) else
       "It is not milder than every training winter." if warmer else "It is colder than every training winter.")))

## 9 · Validation winter 2023/24 (fold 2)

In [ ]:
FACT = {"persistence": BASE["persistence"], "climatology": BASE["climatology"]}
FACT.update({f"{fam} ({fs})": CANDS[fam][name] for (fam, fs), name in chosen.items()})
preds = {}
for name, factory in FACT.items():
    if name == "mlp (weather)":                            # already fitted above
        preds[name] = pd.concat([pd.DataFrame({"target_date": r["score_df"]["target_date"].values, "fold": r["fold"],
                                               "y": r["score_df"]["y"].values, "pred": r["pred"]}) for r in mlp_runs],
                                ignore_index=True)
    else:
        cols = cols_nw if name.endswith("(no weather)") else cols_w
        preds[name] = models.predictions(ablation, cols, factory, cfg)

f2 = {n: d[d["fold"] == 2].set_index("target_date") for n, d in preds.items()}
days = pd.date_range(f2["persistence"].index.min(), f2["persistence"].index.max(), freq="D")
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(days, f2["persistence"]["y"].reindex(days), color="k", lw=1.6, label="observed")
for name, style in (("persistence", ":"), ("ridge (weather)", "-"), ("mlp (weather)", "-")):
    ax.plot(days, f2[name]["pred"].reindex(days), ls=style, lw=1.1, label=name)
ax.axhline(thr, color="#C44E52", ls="--", lw=1, label=f"event threshold {thr:g} µg/m³")
ax.set_ylabel("daily mean PM2.5 (µg/m³)"); ax.legend(fontsize=8, ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.12))
ax.set_title("Validation winter 2023/24: next-day forecasts (gaps = missing observations)")
fig.tight_layout(); plt.show()

## 10 · Save
Validation results only; the test split is not scored.

In [ ]:
out = ROOT / "outputs" / "results"
out.mkdir(parents=True, exist_ok=True)

sel = [base_fold.assign(family="baseline", features="weather")]
sel += [pf.assign(family=fam, features=fs, chosen=pf["candidate"] == chosen[(fam, fs)]) for (fam, fs), (pf, _) in runs.items()]
pd.concat(sel, ignore_index=True).to_csv(out / "forecast_selection.csv", index=False)

wide = None
for name, d in preds.items():
    d = d.rename(columns={"pred": f"pred_{name}"})
    wide = d if wide is None else wide.merge(d.drop(columns="y"), on=["target_date", "fold"])
wide.to_csv(out / "forecast_val_predictions.csv", index=False)

selected = {
    "run_date": pd.Timestamp.now().strftime("%Y-%m-%d"),
    "protocol": "fit on all training months from core.start; score smog-season targets; select on mean MAE of 2 rolling folds",
    "threshold": {"name": thr_name, "ug_m3": thr, "why": thr_why},
    "chosen": {f"{fam} ({fs})": {"config": name, "mean_mae": float(runs[(fam, fs)][1].loc[name, "mean_mae"])}
               for (fam, fs), name in chosen.items()},
    "baselines_mean_mae": final.loc[["persistence", "climatology"], "mae"].to_dict(),
    "features": {"with_weather": cols_w, "without_weather": cols_nw},
    "dropped_features": {"config_drop_columns": drop, "by_missingness_rule": rule_drop,
                         "missing_share_on_rule_base": share.round(4).to_dict()},
}
(out / "forecast_selected.json").write_text(json.dumps(selected, indent=2), encoding="utf-8")
for f in ("forecast_selection.csv", "forecast_val_predictions.csv", "forecast_selected.json"):
    print("Saved", out / f)